The goal of this notebook is to make the two Results figures of the Space Week poster (44 x 33 in) from the Copernicus CHL eddy tables of the canonical experiment.

- Figure 3: change in interior CHL over the observed track, for target and nontarget cyclones and anticyclones on one axis.
- Figure 4: CHL by distance from the eddy center and track age, target and nontarget, cyclones and anticyclones.

Target eddies follow `figures.ipynb`: cyclones formed north of the Gulf Stream axis, or within `NEAR_AXIS_KM` (150 km) of it, and ended south; anticyclones use the reversed rule. The figures are drawn at their printed size on the poster (7.4 x 8.8 in), so the font sizes are poster sizes. Set `SAVE_DIR` to write 300 dpi PNGs.

In [ ]:
from pathlib import Path
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
SAVE_DIR: Path | None = None  # set to a folder to also write 300 dpi PNGs for the poster

NEAR_AXIS_KM = 150
N_AGE_BINS = 5
N_RADIAL_BINS = 10
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
identity_columns = ['polarity', 'track_id']
polarity_names = ('cyclone', 'anticyclone')
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)

# Poster figures are drawn at print size (inches on the 44 x 33 in poster), so text is sized for reading at a few feet
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 20, 'axes.titlesize': 22, 'axes.labelsize': 21, 'xtick.labelsize': 19, 'ytick.labelsize': 19, 'legend.fontsize': 19,
    'axes.linewidth': 1.2, 'xtick.major.width': 1.2, 'ytick.major.width': 1.2, 'xtick.major.size': 6, 'ytick.major.size': 6,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 60, 'savefig.dpi': 300,
})

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['is_target'] = eddy_tracks['movement'].eq(target_class) | (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM) & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['group'] = np.where(eddy_tracks['is_target'], 'target', 'nontarget')
plankton = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet').merge(eddy_tracks[identity_columns + ['group']], on=identity_columns)
plankton_rings = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_rings.parquet')

# Age bin of each eddy-composite and its CHL change from the eddy's first composite
plankton = plankton.sort_values(identity_columns + ['date'])
plankton['age_bin'] = np.minimum((plankton['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
plankton['change'] = plankton['CHL'] - plankton.groupby(identity_columns)['CHL'].transform('first')
n_eddies = plankton.groupby(['group', 'polarity'])['track_id'].nunique()
display(n_eddies.rename('eddies with a CHL composite').to_frame())

## Figure 3: change in CHL over the track, target and nontarget eddies

Each eddy's composites are averaged within each fifth of its observed track, then the eddies are averaged with equal weight. Error bars are 95% bootstrap intervals over eddies. Target eddies are solid, nontarget eddies dashed.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
eddy_bins = plankton.groupby(['group'] + identity_columns + ['age_bin'])['change'].mean().reset_index()
summary_rows = []
for (group, polarity), bins in eddy_bins.groupby(['group', 'polarity']):
    matrix = bins.pivot(index='track_id', columns='age_bin', values='change').reindex(columns=range(N_AGE_BINS)).to_numpy(dtype=float)
    counts = np.isfinite(matrix).sum(axis=0)
    sampled = matrix[rng.integers(0, len(matrix), size=(N_BOOTSTRAP, len(matrix)))]  # (n_bootstrap, n_eddies, n_bins)
    sampled_counts = np.isfinite(sampled).sum(axis=1)
    sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0)
    low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
    summary_rows.append(pd.DataFrame({
        'group': group, 'polarity': polarity, 'age_bin': range(N_AGE_BINS), 'age_midpoint': bin_centers,
        'mean': np.nansum(matrix, axis=0) / counts, 'ci_low': low, 'ci_high': high, 'n_eddies': counts,
    }))
change_summary = pd.concat(summary_rows, ignore_index=True)
display(change_summary.round(4))

group_styles = {'target': {'linestyle': '-', 'marker': 'o', 'linewidth': 3.0}, 'nontarget': {'linestyle': (0, (5, 2.5)), 'marker': 's', 'linewidth': 2.2}}
fig3, ax = cast(tuple[plt.Figure, Axes], plt.subplots(figsize=(7.4, 8.8), layout='constrained'))
ax.axhline(0, color='#999999', linewidth=1.2, zorder=1)
handles = []
for group in ('target', 'nontarget'):
    for polarity in polarity_names:
        result = change_summary.loc[change_summary['group'].eq(group) & change_summary['polarity'].eq(polarity)]
        x = result['age_midpoint']
        color = polarity_colors[polarity]
        ax.errorbar(x, result['mean'], yerr=[result['mean'] - result['ci_low'], result['ci_high'] - result['mean']], fmt='none', ecolor=color, alpha=1 if group == 'target' else 0.6, capsize=4, elinewidth=1.6, capthick=1.6, zorder=2)
        style = group_styles[group]
        ax.plot(x, result['mean'], color=color, linestyle=style['linestyle'], linewidth=style['linewidth'], marker=style['marker'], markersize=9, markeredgecolor='white' if group == 'target' else color, markeredgewidth=1.2 if group == 'target' else 2.0, markerfacecolor=color if group == 'target' else 'white', zorder=3)
        handles.append(Line2D([], [], color=color, linestyle=style['linestyle'], linewidth=style['linewidth'], marker=style['marker'], markersize=9, markeredgewidth=2.0 if group == 'nontarget' else 1.2, markeredgecolor=color if group == 'nontarget' else 'white', markerfacecolor=color if group == 'target' else 'white',
                              label=f'{group.capitalize()} {polarity}s'))
ax.set_xlim(0, 1)
ax.xaxis.set_ticks(np.linspace(0, 1, 6))
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10]))
ax.grid(axis='y', color='#e5e5e5', linewidth=1)
ax.set_axisbelow(True)
ax.set_xlabel('Fraction of observed track')
ax.set_ylabel('$\\Delta$CHL since first observation (mg m$^{-3}$)')
fig3.legend(handles=[handles[0], handles[2], handles[1], handles[3]], loc='outside lower center', ncol=2, handlelength=2.4, columnspacing=1.2, labelspacing=0.3, fontsize=18)
ax.set_ylim(-0.12, 0.18)
if SAVE_DIR is not None:
    fig3.savefig(SAVE_DIR / 'space_week_fig3_chl_change.png', facecolor='white')
plt.show()

## Figure 4: CHL by distance from the eddy center and track age

Rings are 0.2 speed radii wide out to 2 speed radii, so the outer half of each map is the water just outside the eddy. Each cell averages the rings of each eddy within that fifth of its track, then the eddies. All four maps share one color scale.

In [ ]:
radial = plankton_rings.merge(plankton[identity_columns + ['date', 'group', 'age_bin']], on=identity_columns + ['date'])
radial_grid = radial.groupby(['group'] + identity_columns + ['age_bin', 'radial_bin'])['CHL'].mean().groupby(['group', 'polarity', 'age_bin', 'radial_bin']).mean()
chl_norm = Normalize(np.floor(radial_grid.min() / 0.02) * 0.02, np.ceil(radial_grid.max() / 0.02) * 0.02)
heat_cmap = plt.get_cmap('viridis').copy()
heat_cmap.set_bad('#e6e6e6')

fig4 = plt.figure(figsize=(7.4, 8.8))
grid = fig4.add_gridspec(2, 3, left=0.25, right=0.84, bottom=0.13, top=0.93, width_ratios=(1, 1, 0.08), wspace=0.2, hspace=0.18)
for row, group in enumerate(('target', 'nontarget')):
    for col, polarity in enumerate(polarity_names):
        ax = cast(Axes, fig4.add_subplot(grid[row, col]))
        cells = radial_grid.loc[group, polarity].unstack('radial_bin').reindex(index=range(N_AGE_BINS), columns=range(N_RADIAL_BINS)).to_numpy(dtype=float).T
        ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(cells), cmap=heat_cmap, norm=chl_norm, edgecolors='white', linewidth=1.2)
        ax.axhline(1, color='#222222', linewidth=1.8, linestyle=(0, (4, 2.5)), zorder=3)
        if row == 0:
            ax.set_title(f'{polarity.capitalize()}s', pad=8)
        if col == 0:
            ax.set_ylabel('Distance from center\n(speed radii)', fontsize=18)
            ax.text(-0.62, 0.5, group.capitalize(), transform=ax.transAxes, rotation=90, ha='center', va='center', fontsize=22, fontweight='bold')
        ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'] if row == 1 else [])
        ax.yaxis.set_ticks([0, 1, 2], ['0', '1', '2'] if col == 0 else [])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(length=4)
fig4.supxlabel('Fraction of observed track', x=0.52, y=0.015, fontsize=21)
color_bar = fig4.colorbar(ScalarMappable(norm=chl_norm, cmap=heat_cmap), cax=fig4.add_subplot(grid[:, 2]))
color_bar.set_label('CHL (mg m$^{-3}$)')
color_bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10]))
color_bar.outline.set_linewidth(0)
if SAVE_DIR is not None:
    fig4.savefig(SAVE_DIR / 'space_week_fig4_chl_radius_age.png', facecolor='white')
plt.show()